In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/Knowledge-Graph-Enhanced-RAG-for-Technical-Documents")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install datasets spacy networkx matplotlib pandas \
             sentence-transformers faiss-cpu huggingface_hub \
             pyvis requests -q

# spaCy English model for entity extraction
!python -m spacy download en_core_web_sm -q

print("All installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 133.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 98.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
All installed.


In [3]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_Token')
login(token=HF_TOKEN, add_to_git_credential=False)
print("HuggingFace logged in.")

HuggingFace logged in.


In [4]:
from datasets import load_dataset

# technical QA dataset — clean, small, no GPU needed
ds = load_dataset("neural-bridge/rag-dataset-12000", split="train[:200]")

print("Dataset loaded:", len(ds), "samples")
print("Features:", ds.features)
print("\nFirst sample:")
print(ds[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

data/train-00000-of-00001-9df3a936e1f631(…):   0%|          | 0.00/23.1M [00:00<?, ?B/s]

data/test-00000-of-00001-af2a9f454ad1b8a(…):   0%|          | 0.00/5.79M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9600 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2400 [00:00<?, ? examples/s]

Dataset loaded: 200 samples
Features: {'context': Value('string'), 'question': Value('string'), 'answer': Value('string')}

First sample:
{'context': 'Caption: Tasmanian berry grower Nic Hansen showing Macau chef Antimo Merone around his property as part of export engagement activities.\nTHE RISE and rise of the Australian strawberry, raspberry and blackberry industries has seen the sectors redouble their international trade focus, with the release of a dedicated export plan to grow their global presence over the next 10 years.\nDriven by significant grower input, the Berry Export Summary 2028 maps the sectors’ current position, where they want to be, high-opportunity markets and next steps.\nHort Innovation trade manager Jenny Van de Meeberg said the value and volume of raspberry and blackberry exports rose by 100 per cent between 2016 and 2017. She said the Australian strawberry industry experienced similar success with an almost 30 per cent rise in export volume and a 26 per cent ri

In [5]:
import pandas as pd

# check what fields we have
sample = ds[0]
print("Keys:", sample.keys())
print("\nContext snippet:", str(sample.get('context', sample.get('text', '')))[:300])
print("\nQuestion:", sample.get('question', ''))
print("Answer:", sample.get('answer', sample.get('answers', '')))

Keys: dict_keys(['context', 'question', 'answer'])

Context snippet: Caption: Tasmanian berry grower Nic Hansen showing Macau chef Antimo Merone around his property as part of export engagement activities.
THE RISE and rise of the Australian strawberry, raspberry and blackberry industries has seen the sectors redouble their international trade focus, with the release

Question: What is the Berry Export Summary 2028 and what is its purpose?
Answer: The Berry Export Summary 2028 is a dedicated export plan for the Australian strawberry, raspberry, and blackberry industries. It maps the sectors’ current position, where they want to be, high-opportunity markets, and next steps. The purpose of this plan is to grow their global presence over the next 10 years.


In [8]:
# ─── CELL 7: build clean document + QA dataframe ────────────────────────
import pandas as pd

docs = []
qa_pairs = []

for i, sample in enumerate(ds):
    # adapt field names based on Cell 6 output
    context  = sample.get('context',  sample.get('text', ''))
    question = sample.get('question', '')
    answer   = sample.get('answer',   '')
    if isinstance(answer, list):
        answer = answer[0] if answer else ''

    if context and question and answer:
        docs.append({'doc_id': i, 'text': context})
        qa_pairs.append({
            'id':       i,
            'doc_id':   i,
            'question': question,
            'answer':   answer
        })

df_docs = pd.DataFrame(docs).drop_duplicates(subset='text')
df_qa   = pd.DataFrame(qa_pairs)

df_docs.to_csv("data/processed/documents.csv", index=False)
df_qa.to_csv("data/processed/qa_pairs.csv",   index=False)

print(f"Documents: {len(df_docs)}")
print(f"QA pairs:  {len(df_qa)}")
print("\nSample document:")
print(df_docs.iloc[0]['text'][:300])
print("\nSample QA:")
print(df_qa.iloc[0])

Documents: 200
QA pairs:  200

Sample document:
Caption: Tasmanian berry grower Nic Hansen showing Macau chef Antimo Merone around his property as part of export engagement activities.
THE RISE and rise of the Australian strawberry, raspberry and blackberry industries has seen the sectors redouble their international trade focus, with the release

Sample QA:
id                                                          0
doc_id                                                      0
question    What is the Berry Export Summary 2028 and what...
answer      The Berry Export Summary 2028 is a dedicated e...
Name: 0, dtype: object


In [9]:
# ─── CELL 8: entity extraction with spaCy ───────────────────────────────
import spacy
import json

nlp = spacy.load("en_core_web_sm")

def extract_entities(text):
    doc = nlp(text[:1000])  # cap at 1000 chars for speed
    entities = []
    for ent in doc.ents:
        if ent.label_ in ['ORG','PRODUCT','GPE','PERSON',
                           'TECHNOLOGY','WORK_OF_ART','EVENT']:
            entities.append({
                'text':  ent.text.strip(),
                'label': ent.label_
            })
    return entities

# test on first document
sample_text = df_docs.iloc[0]['text']
entities = extract_entities(sample_text)
print("Sample entities extracted:")
for e in entities:
    print(f"  {e['label']:15} {e['text']}")

Sample entities extracted:
  PERSON          Nic Hansen
  GPE             Macau
  PERSON          Antimo Merone
  ORG             RISE
  PERSON          raspberry
  ORG             the Berry Export Summary
  ORG             Hort Innovation
  PERSON          Jenny Van de Meeberg


In [10]:
# ─── CELL 9: extract entities for all documents ─────────────────────────
all_entities = []

for _, row in df_docs.iterrows():
    ents = extract_entities(row['text'])
    for e in ents:
        e['doc_id'] = row['doc_id']
        all_entities.append(e)

df_entities = pd.DataFrame(all_entities)
df_entities.to_csv("data/processed/entities.csv", index=False)

print(f"Total entities extracted: {len(df_entities)}")
print("\nEntity type distribution:")
print(df_entities['label'].value_counts())
print("\nSample entities:")
print(df_entities.head(10))

Total entities extracted: 1609

Entity type distribution:
label
ORG            656
PERSON         552
GPE            322
WORK_OF_ART     47
PRODUCT         22
EVENT           10
Name: count, dtype: int64

Sample entities:
                         text   label  doc_id
0                  Nic Hansen  PERSON       0
1                       Macau     GPE       0
2               Antimo Merone  PERSON       0
3                        RISE     ORG       0
4                   raspberry  PERSON       0
5    the Berry Export Summary     ORG       0
6             Hort Innovation     ORG       0
7        Jenny Van de Meeberg  PERSON       0
8                    Zimbabwe     GPE       1
9  Olschewski • Skat\nProject  PERSON       1


In [12]:
!git config --global user.email "chaitanyamhetre97@email.com"
!git config --global user.name "chaitanyamhetre"

!git config --global user.email "your@email.com"
!git config --global user.name "Your Name"
!git add .
!git commit -m "Dataset loaded, entities extracted"
!git push
print("Pushed to GitHub.")

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean
Enumerating objects: 13, done.
Counting objects: 100% (13/13), done.
Delta compression using up to 2 threads
Compressing objects: 100% (9/9), done.
Writing objects: 100% (12/12), 332.52 KiB | 2.97 MiB/s, done.
Total 12 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/chaitanyamhetre/Knowledge-Graph-Enhanced-RAG-for-Technical-Documents.git
   cc8c26d..6c3b130  main -> main
Pushed to GitHub.
